# 20 kV MV ring — stationary fault analysis

Classical frequency-domain analysis of a 20 kV medium-voltage ring with
20 substations, each connected by 0.5 km of NA2XS(F)2Y 1x300 RM/25 cable.
The shield-earth loop of this cable type carries the per-length values:

| quantity | value | symbol |
| --- | --- | --- |
| shield-earth resistance | 0.25 Ohm/km | R' |
| shield-earth inductance | 1.9 mH/km | L' |
| shield-earth capacitance | 300 nF/km | C' |
| Carson earth-return resistance | 0.05 Ohm/km | R'_mutual |
| phase-shield mutual inductance | 1.85 mH/km | M' |

Each substation is grounded with `R_g = 5 Ohm`. We inject a 1 kA fault
current at `b01`, place a single-phase fault at `b10`, and solve in the
frequency domain to obtain the earth potential rise (EPR), both reduction
factors and the earthing impedance at the fault bus.

The values are representative, not taken from one data sheet;
[Bus and branch types from data sheets](../../howto/types-from-datasheets/)
derives such values for a real cable. The same network is used in the
companion notebook **MV ring (transient)** to study the time-domain response.

In [ ]:
import warnings

import matplotlib.pyplot as plt

import groundinsight as gi

## 1. Cable + bus parameters

Per-length values of the shield-earth loop, plus the lumped totals at the chosen 0.5 km section length.

In [ ]:
# Per-length (per km) shield-earth loop values for NA2XS(F)2Y 1x300 RM/25.
R_PER_KM = 0.25       # ohm / km
L_PER_KM = 1.9e-3     # H / km
C_PER_KM = 300e-9     # F / km
# Carson earth-return mutual values (typical for 50 Hz on this cable type).
R_MUT_PER_KM = 0.05   # ohm / km
M_PER_KM = 1.85e-3    # H / km

SECTION_LENGTH_KM = 0.5
N_BUSES = 20

R_BRANCH = R_PER_KM * SECTION_LENGTH_KM
L_BRANCH = L_PER_KM * SECTION_LENGTH_KM
C_BRANCH = C_PER_KM * SECTION_LENGTH_KM
M_BRANCH = M_PER_KM * SECTION_LENGTH_KM

# Bus grounding resistance (rod / strip / mat equivalent at each substation).
R_BUS = 5.0

# pi-section: the state-space solver of the transient notebook adds C/2 from
# each adjacent branch to the bus -> in the closed ring, every bus accumulates
# one full C_BRANCH from its two halves. The same total goes into the
# frequency-domain bus impedance so that both notebooks model the same physics.
C_BUS_LUMPED = C_BRANCH

print(f"Per-section values (l = {SECTION_LENGTH_KM} km):")
print(f"  R_branch = {R_BRANCH*1e3:.1f} mohm")
print(f"  L_branch = {L_BRANCH*1e3:.2f} mH")
print(f"  C_branch = {C_BRANCH*1e9:.0f} nF")
print(f"  M_branch = {M_BRANCH*1e3:.2f} mH")
print(f"  Bus grounding resistance: R_bus = {R_BUS} ohm")
print(f"  Effective per-bus shunt capacitance after pi-lumping: {C_BUS_LUMPED*1e9:.0f} nF")

## 2. Build the network

* `bus_type` carries the frequency-domain `impedance_formula` — the parallel
  combination `R || 1/(jwC_BUS_LUMPED)` — and, for the transient notebook,
  `R_formula`.
* `branch_type` carries the shield self impedance `R + jwL` and the mutual
  impedance `R_mutual + jwM`, plus the lumped `R_self_formula`,
  `L_self_formula`, `C_self_formula`, `R_mutual_formula` and
  `M_mutual_formula` that the state-space solver of the transient notebook
  reads.

In [ ]:
# Frequency-domain bus impedance: R || (1 / (j*omega*C_lumped)).
Z_bus_formula = (
    f"((rho * 0 + {R_BUS}) / (1 + I * 2 * pi * f * {R_BUS} * {C_BUS_LUMPED}))"
)

bus_type = gi.BusType(
    name="MV_substation",
    description="20 kV substation grounding, 5 ohm; cable C lumped automatically",
    system_type="Substation",
    voltage_level=20.0,
    impedance_formula=Z_bus_formula,
    R_formula=f"rho * 0 + {R_BUS}",
)

Z_self_formula = (
    f"((rho * 0 + {R_PER_KM}) + I * 2 * pi * f * {L_PER_KM}) * l"
)
Z_mutual_formula = (
    f"((rho * 0 + {R_MUT_PER_KM}) + I * 2 * pi * f * {M_PER_KM}) * l"
)
branch_type = gi.BranchType(
    name="NA2XS_300_25",
    description="NA2XS(F)2Y 1x300 RM/25 with Carson mutual coupling",
    grounding_conductor=True,
    self_impedance_formula=Z_self_formula,
    mutual_impedance_formula=Z_mutual_formula,
    R_self_formula=f"({R_PER_KM}) * l",
    L_self_formula=f"({L_PER_KM}) * l",
    C_self_formula=f"({C_PER_KM}) * l",
    R_mutual_formula=f"({R_MUT_PER_KM}) * l",
    M_mutual_formula=f"({M_PER_KM}) * l",
)

net = gi.create_network(name="mv_ring_20", frequencies=[50.0])
for k in range(1, N_BUSES + 1):
    gi.create_bus(name=f"b{k:02d}", type=bus_type, network=net)
for k in range(1, N_BUSES + 1):
    nxt = (k % N_BUSES) + 1
    gi.create_branch(
        name=f"L_{k:02d}_{nxt:02d}",
        type=branch_type,
        from_bus=f"b{k:02d}",
        to_bus=f"b{nxt:02d}",
        length=SECTION_LENGTH_KM,
        network=net,
    )

print(f"Network: {len(net.buses)} buses, {len(net.branches)} branches")

## 3. Stationary fault analysis

1 kA at `b01`, fault at `b10`. The fault current divides between the two
directions around the ring; `run_fault` finds the split by solving the
phase-conductor network. The branch type declares no
`phase_impedance_formula`, so the solver uses the shield impedance as a proxy
for the phase conductor and says so with a `ModelAssumptionWarning`
(`PHASE_PROXY`). In this ring every section is the same cable, so any
impedance proportional to length gives the exact split and the warning has no
consequence here; in a ring that mixes cable and overhead line it would
([Phase currents in rings and meshes](../../explanation/phase-currents/)).

In [ ]:
gi.create_source(name="src", bus="b01", values={50.0: 1000.0 + 0.0j}, network=net)
gi.create_fault(name="F1", bus="b10", network=net)
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    result = gi.run_fault(net, fault="F1")
for warning in caught:
    print(f"{warning.category.__name__} ({warning.message.code})")
print("diagnostics:", [d.code for d in result.diagnostics])

epr_50 = {b.name: b.uepr for b in result.buses}
fig, ax = plt.subplots(figsize=(11, 4))
names = [f"b{k:02d}" for k in range(1, N_BUSES + 1)]
ax.bar(range(1, N_BUSES + 1), [epr_50[n] for n in names], color="tab:blue")
ax.set_xlabel("bus index"); ax.set_ylabel("|EPR| / V (50 Hz, RMS)")
ax.set_title("Stationary EPR profile, 1 kA injected at b01, fault at b10")
ax.set_xticks(range(1, N_BUSES + 1))
ax.grid(True, axis="y", alpha=0.3)
fig.tight_layout(); plt.show()

## 4. Both reduction factors and the earthing impedance

- The **EPR ratio** (`value`, column `reduction_factor`) compares the EPR at
  the fault bus with and without the coupling between phase conductor and
  shields. Every section is the same cable, so it equals the closed form
  $|1 - Z_m/Z_s|$ of one section — the tabulated reduction factor of the
  cable.
- The **reduction factor of EN 50522** (`value_current`, column
  `reduction_factor_current`) is the share of the fault current that returns
  through the soil, $|I_E|/|3I_0|$.
- $Z_E$ is the earthing impedance at the fault bus: the driving-point
  impedance of all electrodes and shields seen from `b10`.

[Reduction factors and earthing impedance](../../explanation/reduction-factors/)
explains why the two factors differ.

In [ ]:
import cmath

factor = result.reduction_factor
omega = 2 * cmath.pi * 50.0
z_s = R_PER_KM + 1j * omega * L_PER_KM
z_m = R_MUT_PER_KM + 1j * omega * M_PER_KM

z_e = complex(factor.z_earthing[50.0].real, factor.z_earthing[50.0].imag)
print(f"EPR ratio  (value)          = {factor.value[50.0]:.3f}   closed form |1 - Z_m/Z_s| = {abs(1 - z_m / z_s):.3f}")
print(f"EN 50522 r (value_current)  = {factor.value_current[50.0]:.3f}")
print(f"Z_E at the fault bus (b10)  = {abs(z_e):.3f} ohm at {cmath.phase(z_e) * 180 / cmath.pi:.1f} deg")
print(f"U_E = EPR at b10            = {epr_50['b10']:.1f} V")